In [1]:
import duckdb
import os
import json
import pandas as pd

def get_minutes_played(match_id):
    path = os.path.expanduser(f"~/projects/statsbomb-data/data/events/{match_id}.json")

    with open(path) as f:
        raw_events = json.load(f)

    starting_xi_events = [e for e in raw_events if e['type']['name'] == 'Starting XI']
    sub_events = [e for e in raw_events if e['type']['name'] == 'Substitution']

    start_minutes = {}
    player_names = {}

    for lineup_event in starting_xi_events:
        for player_entry in lineup_event['tactics']['lineup']:
            pid = player_entry['player']['id']
            start_minutes[pid] = 0
            player_names[pid] = player_entry['player']['name']

    end_minutes = {}
    for sub in sub_events:
        off_id = sub['player']['id']
        on_id = sub['substitution']['replacement']['id']
        end_minutes[off_id] = sub['minute']
        start_minutes[on_id] = sub['minute']
        player_names[on_id] = sub['substitution']['replacement']['name']

    final_minute = max(e['minute'] for e in raw_events)

    minutes_played = {}
    for pid in start_minutes:
        player_end = end_minutes.get(pid, final_minute)
        minutes_played[pid] = player_end - start_minutes[pid]

    df = pd.DataFrame.from_dict(minutes_played, orient='index')
    df = df.reset_index()
    df.columns = ['player_id', 'minutes_played']
    df['player_name'] = df['player_id'].map(player_names)
    df['match_id'] = match_id
    return df

test2 = get_minutes_played(3754217)
print(test2.shape)
test2.head()

(28, 4)


,player_id,minutes_played,player_name,match_id
0,3339,95,Asmir Begović,3754217
1,5594,95,Branislav Ivanović,3754217
2,3456,95,Kurt Happy Zouma,3754217
3,3645,95,Gary Cahill,3754217
4,3957,95,César Azpilicueta Tanco,3754217


In [2]:
##players_id = [e for e in starting_xi_events if e[0]['tactics']['lineup'] == "id"]
##players_name = [e for e in starting_xi_events if e[0]['tactics']['lineup'] == "name"]

##print(players_id)
##print(players_name)

##incorrect